In [2]:
## Import Libraries
import pandas as pd

In [11]:
# Load the cleaned SEC Gig datasets
company_financials_10K_10Q = pd.read_csv('../../data/sec_10K-10Q/company_financials_10K-10Q.csv')
company_financials_10K_only = pd.read_csv('../../data/sec_10K-10Q/company_financials_10K-only.csv')

# Load the cleaned Yahoo Finance Gig datasets
yfinance_data_quarterly = pd.read_csv('../../data/yfinance/gig_data_quarterly.csv')
yfinance_data_annual = pd.read_csv('../../data/yfinance/gig_data_annually.csv')

In [ ]:
# Combine company financials with stock price data for correlation analysis
def combine_financials_with_stock_prices_quarterly(financials_df, stock_prices_df):
    # Map fiscal period strings to integer quarters
    fiscal_to_quarter = {
        'Q1': 1, 'Q2': 2, 'Q3': 3, 'Q4': 4,
        'FY': 4  # Treat annual filings as Q4
    }
    
    # yfinance colums names to lowercase to match financials_df
    stock_prices_df.columns = stock_prices_df.columns.str.lower()
    
    df_fin = financials_df.copy()
    df_fin['quarter'] = df_fin['fiscal_period'].map(fiscal_to_quarter)
    df_fin['year'] = df_fin['fiscal_year'].astype(int)

    # Normalize ticker casing to match yfinance
    df_fin['ticker'] = df_fin['ticker'].str.upper()

    combined_df = pd.merge(
        df_fin,
        stock_prices_df,
        on=['ticker', 'year', 'quarter'],
        how='left'
    )
    
    # remove cols that end with '_y' (from yfinance) that are duplicates of financials_df columns
    combined_df = combined_df[[col for col in combined_df.columns if not col.endswith('_y')]]
    
    # correct any remaining '_x' suffixes from financials_df
    combined_df.columns = [col.replace('_x', '') for col in combined_df.columns]
    
    # numeric cols last to avoid merge issues
    numeric_cols = ['Revenue', 'NetIncomeLoss', 'OperatingIncomeLoss', 'Assets', 'Liabilities', 'StockholdersEquity', 'IncomeTaxExpenseBenefit', 'CommonStockSharesOutstanding']
    lag_cols = [col for col in combined_df.columns if col.endswith('_lag1')]
    numeric_cols += lag_cols
    other_cols = [col for col in combined_df.columns if col not in numeric_cols]
    combined_df = combined_df[other_cols + numeric_cols]

    return combined_df

# Combine datasets
combined_quarterly = combine_financials_with_stock_prices_quarterly(company_financials_10K_10Q, yfinance_data_quarterly)
combined_quarterly


,filed_date,fiscal_year,form,fiscal_period,company,ticker,category,quarter,year,date,...,Assets_lag1,CommonStockSharesOutstanding_lag1,IncomeTaxExpenseBenefit_lag1,Liabilities_lag1,NetIncomeLoss_lag1,OperatingIncomeLoss_lag1,StockholdersEquity_lag1,Revenue_lag1,close_lag1,volume_lag1
0,2021-03-05,2020,10-K,FY,DoorDash,DASH,gig,4,2020,2020-12-31,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-05-14,2021,10-Q,Q1,DoorDash,DASH,gig,1,2021,2021-03-31,...,1.732000e+09,3.999909e+08,0.0,5.500000e+08,-204000000.0,-210000000.0,-1.980000e+08,2.910000e+08,142.750000,2637300.0
2,2021-08-13,2021,10-Q,Q2,DoorDash,DASH,gig,2,2021,2021-06-30,...,6.353000e+09,3.999909e+08,1000000.0,1.653000e+09,-129000000.0,-123000000.0,-1.082000e+09,3.620000e+08,131.130005,2939200.0
3,2021-11-10,2021,10-Q,Q3,DoorDash,DASH,gig,3,2021,2021-09-30,...,6.353000e+09,3.999909e+08,1000000.0,1.653000e+09,-129000000.0,-96000000.0,-1.082000e+09,1.037000e+09,178.330002,4880300.0
4,2022-03-01,2021,10-K,FY,DoorDash,DASH,gig,4,2021,2021-12-31,...,6.353000e+09,3.999909e+08,2000000.0,1.653000e+09,-129000000.0,-131000000.0,-1.082000e+09,1.916000e+09,205.979996,1565500.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296,2025-02-13,2024,10-K,FY,Upwork,UPWK,gig,4,2024,2024-12-31,...,1.037541e+09,1.372728e+08,3547000.0,6.564660e+08,29513000.0,-19688000.0,2.488790e+08,5.052020e+08,10.450000,994700.0
297,2025-05-05,2025,10-Q,Q1,Upwork,UPWK,gig,1,2025,2025-03-31,...,1.037541e+09,1.372728e+08,536000.0,6.564660e+08,-89885000.0,-92624000.0,2.595170e+08,6.183180e+08,16.350000,1107900.0
298,2025-08-06,2025,10-Q,Q2,Upwork,UPWK,gig,2,2025,2025-06-30,...,1.211613e+09,1.353485e+08,1329000.0,6.362360e+08,18442000.0,13049000.0,3.810750e+08,1.909370e+08,13.050000,1488900.0
299,2025-11-04,2025,10-Q,Q3,Upwork,UPWK,gig,3,2025,2025-09-30,...,1.211613e+09,1.353485e+08,2510000.0,6.362360e+08,40662000.0,30830000.0,3.810750e+08,3.840660e+08,13.440000,2960800.0


In [24]:
# save combined dataset
combined_quarterly.to_csv('../../data/combined_sec_yfin/sec_yfin_quarterly.csv', index=False)

combined_quarterly.columns

Index(['filed_date', 'fiscal_year', 'form', 'fiscal_period', 'company',
       'ticker', 'category', 'quarter', 'year', 'date', 'industrykey',
       'sectorkey', 'close', 'volume', 'Revenue', 'NetIncomeLoss',
       'OperatingIncomeLoss', 'Assets', 'Liabilities', 'StockholdersEquity',
       'IncomeTaxExpenseBenefit', 'CommonStockSharesOutstanding',
       'Assets_lag1', 'CommonStockSharesOutstanding_lag1',
       'IncomeTaxExpenseBenefit_lag1', 'Liabilities_lag1',
       'NetIncomeLoss_lag1', 'OperatingIncomeLoss_lag1',
       'StockholdersEquity_lag1', 'Revenue_lag1', 'close_lag1', 'volume_lag1'],
      dtype='str')

In [25]:
# Combine company financials with stock price data for correlation analysis
def combine_financials_with_stock_prices_annually(financials_df, stock_prices_df):
    # yfinance colums names to lowercase to match financials_df
    stock_prices_df.columns = stock_prices_df.columns.str.lower()
    
    df_fin = financials_df.copy()
    df_fin['year'] = df_fin['fiscal_year'].astype(int)

    # Normalize ticker casing to match yfinance
    df_fin['ticker'] = df_fin['ticker'].str.upper()

    combined_df = pd.merge(
        df_fin,
        stock_prices_df,
        on=['ticker', 'year'],
        how='left'
    )
    
    # remove cols that end with '_y' (from yfinance) that are duplicates of financials_df columns
    combined_df = combined_df[[col for col in combined_df.columns if not col.endswith('_y')]]
    
    # correct any remaining '_x' suffixes from financials_df
    combined_df.columns = [col.replace('_x', '') for col in combined_df.columns]
    
    # numeric cols last to avoid merge issues
    numeric_cols = ['Revenue', 'NetIncomeLoss', 'OperatingIncomeLoss', 'Assets', 'Liabilities', 'StockholdersEquity', 'IncomeTaxExpenseBenefit', 'CommonStockSharesOutstanding']
    lag_cols = [col for col in combined_df.columns if col.endswith('_lag1')]
    numeric_cols += lag_cols
    other_cols = [col for col in combined_df.columns if col not in numeric_cols]
    combined_df = combined_df[other_cols + numeric_cols]

    return combined_df

# Combine datasets
combined_annual = combine_financials_with_stock_prices_annually(company_financials_10K_only, yfinance_data_annual)
combined_annual

,filed_date,fiscal_year,form,fiscal_period,company,ticker,category,year,date,industrykey,...,Assets_lag1,CommonStockSharesOutstanding_lag1,IncomeTaxExpenseBenefit_lag1,Liabilities_lag1,NetIncomeLoss_lag1,OperatingIncomeLoss_lag1,StockholdersEquity_lag1,Revenue_lag1,close_lag1,volume_lag1
0,2021-03-05,2020,10-K,FY,DoorDash,DASH,gig,2020,2020-12-31,internet-retail,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-03-01,2021,10-K,FY,DoorDash,DASH,gig,2021,2021-12-31,internet-retail,...,6.353000e+09,3.999909e+08,2000000.0,1.653000e+09,-129000000.0,-131000000.0,-1.082000e+09,1.916000e+09,142.750000,2637300.0
2,2023-02-27,2022,10-K,FY,DoorDash,DASH,gig,2022,2022-12-31,internet-retail,...,6.809000e+09,3.999909e+08,3000000.0,2.142000e+09,-110000000.0,-298000000.0,4.700000e+09,3.588000e+09,148.899994,2297900.0
3,2024-02-20,2023,10-K,FY,DoorDash,DASH,gig,2023,2023-12-31,internet-retail,...,9.789000e+09,3.999909e+08,-14000000.0,3.021000e+09,-167000000.0,-754000000.0,4.667000e+09,4.765000e+09,48.820000,3193300.0
4,2025-02-14,2024,10-K,FY,DoorDash,DASH,gig,2024,2024-12-31,internet-retail,...,1.083900e+10,3.999909e+08,14000000.0,4.026000e+09,-161000000.0,-490000000.0,6.754000e+09,6.332000e+09,98.889999,2525700.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,2022-02-15,2021,10-K,FY,Upwork,UPWK,gig,2021,2021-12-31,internet-content-information,...,5.292270e+08,1.247952e+08,57000.0,2.299170e+08,-23792000.0,-23064000.0,2.594240e+08,2.674750e+08,34.520000,1485200.0
74,2023-02-16,2022,10-K,FY,Upwork,UPWK,gig,2022,2022-12-31,internet-content-information,...,1.081061e+09,1.291305e+08,59000.0,8.215440e+08,-33684000.0,-32409000.0,2.993100e+08,3.659410e+08,34.160000,672300.0
75,2024-02-15,2023,10-K,FY,Upwork,UPWK,gig,2023,2023-12-31,internet-content-information,...,1.080245e+09,1.323683e+08,96000.0,8.313660e+08,-73385000.0,-72142000.0,2.595170e+08,4.568760e+08,10.440000,1120800.0
76,2025-02-13,2024,10-K,FY,Upwork,UPWK,gig,2024,2024-12-31,internet-content-information,...,1.037541e+09,1.372728e+08,3547000.0,6.564660e+08,29513000.0,-19688000.0,2.488790e+08,5.052020e+08,14.870000,1535700.0


In [26]:
# save combined dataset
combined_annual.to_csv('../../data/combined_sec_yfin/sec_yfin_annual.csv', index=False)

combined_annual.columns

Index(['filed_date', 'fiscal_year', 'form', 'fiscal_period', 'company',
       'ticker', 'category', 'year', 'date', 'industrykey', 'sectorkey',
       'close', 'volume', 'Revenue', 'NetIncomeLoss', 'OperatingIncomeLoss',
       'Assets', 'Liabilities', 'StockholdersEquity',
       'IncomeTaxExpenseBenefit', 'CommonStockSharesOutstanding',
       'Assets_lag1', 'CommonStockSharesOutstanding_lag1',
       'IncomeTaxExpenseBenefit_lag1', 'Liabilities_lag1',
       'NetIncomeLoss_lag1', 'OperatingIncomeLoss_lag1',
       'StockholdersEquity_lag1', 'Revenue_lag1', 'close_lag1', 'volume_lag1'],
      dtype='str')